# 4-layer and 6-layer MLPs without dropout, with the PCA-space pseudo-labels

Follow-up to `phase1_deep_mlp_nodropout_pseudo_labels.ipynb`. The 6-layer network and its training are unchanged. A 4-layer network is added,
and the pseudo-labels for the 72 clean unlabelled rows now come from **Section 8 of `pca_explained_variance.ipynb` on `vt/pre-processing`**.
Those classifiers were fitted directly on the 118-D PCA coordinates, not on a 2-D map.

| | Phase 1 baseline (reference) | 4-layer | 6-layer |
|---|---|---|---|
| Hidden layers | 512 → 256 | **512 → 256 → 256 → 128** | 512 → 512 → 256 → 256 → 128 → 128 |
| Dropout (hidden / input) | 0.5 / 0.2 | **0 / 0** | 0 / 0 |
| Weight decay (AdamW) | 0.05 | 1.0 | 1.0 (the previous notebook's sweep; the spread over 0.01–1.0 was only 0.009) |
| Unchanged | GELU, lr 1e-3 with cosine decay, batch 64, 40 epochs, logit-adjusted cross-entropy (τ = 1), standardization fitted on the training part | same | same |

**The 3 arms**
- `none`: the 1284 labelled rows only.
- `5NN_cosine_pca118`: plus the 72 clean unlabelled rows, labelled by distance-weighted 5-NN (cosine) in the 118-D PCA space.
- `logreg_pca118`: plus the same 72 rows, labelled by logistic regression (C = 0.01) in the 118-D PCA space.

**Test-set accuracy cannot be computed here,** because `Test_Data.csv` has no labels. Section 4 writes one submission file per (depth, arm)
for the Kaggle leaderboard. The CV numbers below use the fixed Phase 0 folds (5 folds × 3 seeds), like every other entry in the log.

**Caveats**
- The 4-layer net reuses the 6-layer's weight decay without its own sweep.
- The pseudo-label leak described in `phase1_pseudo_labels.ipynb` still applies: the 72 labels were made using all 136 clean labelled rows,
  including the clean rows of each validation fold. Treat gains that show up only in clean-group accuracy with suspicion.

**Where it runs:** this notebook imports `ie643/` and reads `folds/cv_folds_v1.json`, which live on `pensieve_branch`. Run it from a `pensieve_branch` checkout placed inside the `vt/pre-processing` checkout (as `Pensieve_Branch/`), so that `../pseudo_labels/` resolves. The copy on `vt/pre-processing` is kept for its results.

## 1. Setup

In [1]:
import json
import time
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from ie643 import cv, data, mlp
from ie643.data import N_CLASSES

pd.set_option("display.precision", 3)
print("torch", torch.__version__, "| CPU threads", torch.get_num_threads())

d = data.load()
groups, fold = cv.load_folds(d.train_id)
lab = d.labelled
X, y, g, f = d.E[lab], d.y[lab], groups[lab], fold[lab]     # labelled rows only, as in Phase 1
print("labelled rows:", len(y))

BASE_2L = mlp.Config(hidden=(512, 256), dropout=0.5, input_dropout=0.2, lr=1e-3, weight_decay=0.05,
                     epochs=40, batch_size=64, loss="logit_adjusted", tau=1.0)    # Phase 1 final config
NO_DROP = dict(dropout=0.0, input_dropout=0.0, weight_decay=1.0)
DEPTHS = {
    "4-layer": replace(BASE_2L, hidden=(512, 256, 256, 128), **NO_DROP),
    "6-layer": replace(BASE_2L, hidden=(512, 512, 256, 256, 128, 128), **NO_DROP),
}
SEEDS = (0, 1, 2)
FINAL_SEEDS = (0, 1, 2, 3, 4)

n_params = lambda cfg: sum(p.numel() for p in mlp.MLP(np.zeros(768), np.ones(768), cfg).parameters())
print(f"parameters: 2-layer {n_params(BASE_2L):,} | " + " | ".join(f"{k} {n_params(c):,}" for k, c in DEPTHS.items()))

torch 2.12.1 | CPU threads 10
labelled rows: 1284
parameters: 2-layer 527,883 | 4-layer 625,163 | 6-layer 904,331


## 2. The 72 extra rows (PCA-space pseudo-labels)

Read from `pseudo_labels/` one level above this folder (the `vt/pre-processing` checkout).

In [2]:
PSEUDO_DIR = data.REPO_ROOT.parent / "pseudo_labels"
VARIANTS = ["5NN_cosine_pca118", "logreg_pca118"]
ARMS = ["none"] + VARIANTS
assert PSEUDO_DIR.is_dir(), f"pseudo-label folder not found: {PSEUDO_DIR}"

row_of = {int(s): i for i, s in enumerate(d.train_id)}
extra = {"none": (np.zeros((0, d.E.shape[1]), dtype=d.E.dtype), np.zeros(0, dtype=y.dtype))}
for v in VARIANTS:
    p = pd.read_csv(PSEUDO_DIR / f"pseudo_labels_clean_unlabelled_{v}.csv")
    rows = np.array([row_of[int(s)] for s in p["sample_id"]])
    assert len(rows) == 72 and len(set(rows)) == 72
    assert (d.y[rows] == -1).all() and (~d.deformed[rows]).all(), "expected clean unlabelled rows only"
    assert p["predicted_label"].between(0, N_CLASSES - 1).all()
    extra[v] = (d.E[rows], p["predicted_label"].to_numpy().astype(y.dtype))
print({v: len(extra[v][1]) for v in ARMS})
print("the two label sets agree on", int((extra[VARIANTS[0]][1] == extra[VARIANTS[1]][1]).sum()), "of 72 rows")

{'none': 0, '5NN_cosine_pca118': 72, 'logreg_pca118': 72}
the two label sets agree on 67 of 72 rows


In [3]:
def run_cv_with_extra(X, y, g, f, X_extra, y_extra, cfg, seeds):
    # Same as ie643.mlp.run_cv, with the extra rows appended to the training part of every fold
    metrics = []
    oof = np.zeros((len(y), N_CLASSES))
    for seed in seeds:
        for k in range(cv.N_FOLDS):
            tr, va = f != k, f == k
            model, _ = mlp.fit(np.vstack([X[tr], X_extra]), np.concatenate([y[tr], y_extra]), cfg, seed=seed)
            p = mlp.predict_proba(model, X[va])
            oof[va] += p / len(seeds)
            metrics.append({"seed": seed, "fold": k, **cv.evaluate(y[va], p.argmax(1), g[va])})
    return pd.DataFrame(metrics), oof


COLS = ["balanced accuracy", "accuracy", "accuracy: clean", "accuracy: deformed", "accuracy: far"]

def summarize(metrics, by):
    mean = metrics.groupby(by, sort=False)[COLS].mean()
    mean["balanced accuracy, std over runs"] = metrics.groupby(by, sort=False)["balanced accuracy"].std()
    return mean


def paired(metrics, by, ref, others, metrics_list=("balanced accuracy", "accuracy", "accuracy: deformed", "accuracy: far")):
    piv = metrics.pivot_table(index=["seed", "fold"], columns=by, values=list(metrics_list))
    rows = []
    for o in others:
        for m in metrics_list:
            diff = piv[(m, o)] - piv[(m, ref)]
            rows.append({by: o, "metric": m, f"mean difference vs {ref}": diff.mean(), "std": diff.std(),
                         "runs where it is higher": f"{(diff > 0).sum()}/{len(diff)}"})
    return pd.DataFrame(rows).set_index([by, "metric"])

## 3. Cross-validation: both depths × 3 arms, plus the 2-layer Phase 1 baseline

Same folds and seeds for every run, so all comparisons are run by run (paired).

In [4]:
cv_metrics, cv_oof, t_all = [], {}, time.time()
m, _ = run_cv_with_extra(X, y, g, f, *extra["none"], BASE_2L, SEEDS)
cv_metrics.append(m.assign(arm="2-layer: none (Phase 1)"))
for depth, cfg in DEPTHS.items():
    for v in ARMS:
        t0 = time.time()
        m, oof = run_cv_with_extra(X, y, g, f, *extra[v], cfg, SEEDS)
        cv_metrics.append(m.assign(arm=f"{depth}: {v}"))
        cv_oof[(depth, v)] = oof
        print(f"{depth} {v:20s} {len(m)} runs in {time.time() - t0:.0f} s")
cv_metrics = pd.concat(cv_metrics, ignore_index=True)
print(f"total {time.time() - t_all:.0f} s")

display(summarize(cv_metrics, "arm"))

4-layer none                 15 runs in 26 s


4-layer 5NN_cosine_pca118    15 runs in 27 s


4-layer logreg_pca118        15 runs in 27 s


6-layer none                 15 runs in 36 s


6-layer 5NN_cosine_pca118    15 runs in 39 s


6-layer logreg_pca118        15 runs in 39 s
total 218 s


,balanced accuracy,accuracy,accuracy: clean,accuracy: deformed,accuracy: far,"balanced accuracy, std over runs"
arm,,,,,,
2-layer: none (Phase 1),0.682,0.775,0.927,0.781,0.592,0.032
4-layer: none,0.639,0.746,0.923,0.755,0.522,0.043
4-layer: 5NN_cosine_pca118,0.649,0.747,0.925,0.757,0.513,0.035
4-layer: logreg_pca118,0.655,0.750,0.930,0.760,0.515,0.041
6-layer: none,0.607,0.724,0.884,0.735,0.499,0.035
6-layer: 5NN_cosine_pca118,0.617,0.727,0.891,0.733,0.533,0.030
6-layer: logreg_pca118,0.616,0.728,0.906,0.732,0.538,0.050


In [5]:
print("paired against the 2-layer Phase 1 baseline:")
display(paired(cv_metrics, "arm", "2-layer: none (Phase 1)", [f"{dp}: {v}" for dp in DEPTHS for v in ARMS],
               metrics_list=("balanced accuracy", "accuracy")))
for depth in DEPTHS:
    print(f"paired against '{depth}: none' (effect of the pseudo-labels at this depth):")
    display(paired(cv_metrics, "arm", f"{depth}: none", [f"{depth}: {v}" for v in VARIANTS]))
print("4-layer vs 6-layer, same arm:")
display(pd.concat([paired(cv_metrics, "arm", f"6-layer: {v}", [f"4-layer: {v}"], metrics_list=("balanced accuracy", "accuracy"))
                     .rename(columns={f"mean difference vs 6-layer: {v}": "mean difference, 4-layer minus 6-layer"})
                   for v in ARMS]))

paired against the 2-layer Phase 1 baseline:


mean difference vs 2-layer: none (Phase 1)  \
arm                        metric                                                          
4-layer: none              balanced accuracy                                      -0.044   
                           accuracy                                               -0.029   
4-layer: 5NN_cosine_pca118 balanced accuracy                                      -0.034   
                           accuracy                                               -0.028   
4-layer: logreg_pca118     balanced accuracy                                      -0.028   
                           accuracy                                               -0.025   
6-layer: none              balanced accuracy                                      -0.075   
                           accuracy                                               -0.052   
6-layer: 5NN_cosine_pca118 balanced accuracy                                      -0.065   
                           accuracy                                               -0.049   
6-layer: logreg_pca118     balanced accuracy                                      -0.067   
                           accuracy                                               -0.047   

                                                std runs where it is higher  
arm                        metric                                            
4-layer: none              balanced accuracy  0.031                    1/15  
                           accuracy           0.019                    1/15  
4-layer: 5NN_cosine_pca118 balanced accuracy  0.025                    2/15  
                           accuracy           0.016                    0/15  
4-layer: logreg_pca118     balanced accuracy  0.028                    3/15  
                           accuracy           0.021                    1/15  
6-layer: none              balanced accuracy  0.035                    1/15  
                           accuracy           0.021                    0/15  
6-layer: 5NN_cosine_pca118 balanced accuracy  0.037                    0/15  
                           accuracy           0.026                    0/15  
6-layer: logreg_pca118     balanced accuracy  0.050                    1/15  
                           accuracy           0.033                    1/15

paired against '4-layer: none' (effect of the pseudo-labels at this depth):


mean difference vs 4-layer: none  \
arm                        metric                                                 
4-layer: 5NN_cosine_pca118 balanced accuracy                              0.010   
                           accuracy                                       0.001   
                           accuracy: deformed                             0.002   
                           accuracy: far                                 -0.010   
4-layer: logreg_pca118     balanced accuracy                              0.016   
                           accuracy                                       0.004   
                           accuracy: deformed                             0.005   
                           accuracy: far                                 -0.007   

                                                 std runs where it is higher  
arm                        metric                                             
4-layer: 5NN_cosine_pca118 balanced accuracy   0.020                   11/15  
                           accuracy            0.015                    7/15  
                           accuracy: deformed  0.016                    9/15  
                           accuracy: far       0.056                    6/15  
4-layer: logreg_pca118     balanced accuracy   0.032                    9/15  
                           accuracy            0.020                    8/15  
                           accuracy: deformed  0.020                    7/15  
                           accuracy: far       0.052                    4/15

paired against '6-layer: none' (effect of the pseudo-labels at this depth):


mean difference vs 6-layer: none  \
arm                        metric                                                 
6-layer: 5NN_cosine_pca118 balanced accuracy                              0.011   
                           accuracy                                       0.003   
                           accuracy: deformed                            -0.002   
                           accuracy: far                                  0.034   
6-layer: logreg_pca118     balanced accuracy                              0.009   
                           accuracy                                       0.004   
                           accuracy: deformed                            -0.003   
                           accuracy: far                                  0.039   

                                                 std runs where it is higher  
arm                        metric                                             
6-layer: 5NN_cosine_pca118 balanced accuracy   0.024                   10/15  
                           accuracy            0.021                    7/15  
                           accuracy: deformed  0.025                    5/15  
                           accuracy: far       0.050                    9/15  
6-layer: logreg_pca118     balanced accuracy   0.028                    8/15  
                           accuracy            0.019                   10/15  
                           accuracy: deformed  0.021                    6/15  
                           accuracy: far       0.071                   11/15

4-layer vs 6-layer, same arm:


mean difference, 4-layer minus 6-layer  \
arm                        metric                                                      
4-layer: none              balanced accuracy                                   0.032   
                           accuracy                                            0.022   
4-layer: 5NN_cosine_pca118 balanced accuracy                                   0.031   
                           accuracy                                            0.020   
4-layer: logreg_pca118     balanced accuracy                                   0.039   
                           accuracy                                            0.022   

                                                std runs where it is higher  
arm                        metric                                            
4-layer: none              balanced accuracy  0.038                   12/15  
                           accuracy           0.022                   13/15  
4-layer: 5NN_cosine_pca118 balanced accuracy  0.039                   12/15  
                           accuracy           0.028                   12/15  
4-layer: logreg_pca118     balanced accuracy  0.046                   12/15  
                           accuracy           0.030                   10/15

In [6]:
recall = pd.DataFrame({f"{dp}: {v}": cv.per_class_recall(y, cv_oof[(dp, v)].argmax(1)) for dp in DEPTHS for v in ARMS})
recall.index.name = "class"
display(recall.T)

class,0,1,2,3,4,5,6,7,8,9,10
4-layer: none,0.933,0.565,0.542,0.637,0.689,0.696,0.738,0.701,0.44,0.689,0.575
4-layer: 5NN_cosine_pca118,0.929,0.587,0.562,0.637,0.676,0.696,0.738,0.722,0.46,0.705,0.550
4-layer: logreg_pca118,0.936,0.543,0.542,0.648,0.689,0.722,0.738,0.732,0.50,0.689,0.525
6-layer: none,0.938,0.500,0.500,0.604,0.676,0.643,0.730,0.660,0.48,0.674,0.425
6-layer: 5NN_cosine_pca118,0.938,0.565,0.521,0.637,0.676,0.635,0.716,0.660,0.48,0.644,0.425
6-layer: logreg_pca118,0.938,0.565,0.500,0.604,0.703,0.652,0.738,0.680,0.46,0.659,0.400


## 4. Final models and submission files

For each (depth, arm): 5 seeds trained on all 1284 labelled rows plus that arm's extra rows, with test probabilities averaged.
- Weights go to `checkpoints/deep<4|6>_nodrop_<arm>/`.
- Submissions go to `submissions/deep<4|6>_nodrop_mlp_logit_adjusted_<arm>.csv`.

Both folders are git-ignored.

In [7]:
test_proba, test_pred, t_all = {}, {}, time.time()
Path("submissions").mkdir(exist_ok=True)
for depth, cfg in DEPTHS.items():
    tag = f"deep{len(cfg.hidden)}_nodrop"
    for v in ARMS:
        X_extra, y_extra = extra[v]
        X_fit, y_fit = np.vstack([X, X_extra]), np.concatenate([y, y_extra])
        models = mlp.fit_ensemble(X_fit, y_fit, cfg, seeds=FINAL_SEEDS)
        test_proba[(depth, v)] = mlp.ensemble_proba(models, d.F)
        test_pred[(depth, v)] = test_proba[(depth, v)].argmax(1)

        ckpt_dir = Path("checkpoints") / f"{tag}_{v}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        for s, mdl in zip(FINAL_SEEDS, models):
            torch.save(mdl.state_dict(), ckpt_dir / f"seed{s}.pt")
        (ckpt_dir / "config.json").write_text(json.dumps({**asdict(cfg), "seeds": FINAL_SEEDS, "pseudo_labels": v,
                                                          "training rows": int(len(y_fit))}, indent=1))

        sub = pd.DataFrame({"ID": d.test_id.astype(int), "Label": test_pred[(depth, v)].astype(int)})
        assert len(sub) == 490 and sub["ID"].is_unique and sub["Label"].between(0, N_CLASSES - 1).all()
        sub_path = Path("submissions") / f"{tag}_mlp_logit_adjusted_{v}.csv"
        sub.to_csv(sub_path, index=False)
        print(f"{depth} {v:20s} {len(y_fit)} training rows -> wrote {sub_path}")
print(f"total {time.time() - t_all:.0f} s")

4-layer none                 1284 training rows -> wrote submissions/deep4_nodrop_mlp_logit_adjusted_none.csv


4-layer 5NN_cosine_pca118    1356 training rows -> wrote submissions/deep4_nodrop_mlp_logit_adjusted_5NN_cosine_pca118.csv


4-layer logreg_pca118        1356 training rows -> wrote submissions/deep4_nodrop_mlp_logit_adjusted_logreg_pca118.csv


6-layer none                 1284 training rows -> wrote submissions/deep6_nodrop_mlp_logit_adjusted_none.csv


6-layer 5NN_cosine_pca118    1356 training rows -> wrote submissions/deep6_nodrop_mlp_logit_adjusted_5NN_cosine_pca118.csv


6-layer logreg_pca118        1356 training rows -> wrote submissions/deep6_nodrop_mlp_logit_adjusted_logreg_pca118.csv
total 77 s


In [8]:
share = lambda p: np.bincount(p, minlength=N_CLASSES) / len(p)
mix = pd.DataFrame({"labelled train (true labels)": share(y),
                    **{f"test predictions: {dp}: {v}": share(test_pred[(dp, v)]) for dp in DEPTHS for v in ARMS}})
mix.index.name = "class"
display(mix.T.round(3))

keys = [(dp, v) for dp in DEPTHS for v in ARMS]
disagree = pd.DataFrame([[int((test_pred[a] != test_pred[b]).sum()) for b in keys] for a in keys],
                        index=[f"{dp}: {v}" for dp, v in keys], columns=[f"{dp}: {v}" for dp, v in keys])
print("number of the 490 test rows where two submissions differ:")
display(disagree)

class,0,1,2,3,4,5,6,7,8,9,10
labelled train (true labels),0.350,0.036,0.037,0.071,0.058,0.090,0.110,0.076,0.039,0.103,0.031
test predictions: 4-layer: none,0.118,0.045,0.073,0.106,0.084,0.118,0.110,0.092,0.094,0.096,0.063
test predictions: 4-layer: 5NN_cosine_pca118,0.114,0.045,0.076,0.106,0.086,0.116,0.116,0.088,0.096,0.090,0.067
test predictions: 4-layer: logreg_pca118,0.122,0.045,0.069,0.102,0.088,0.116,0.106,0.094,0.094,0.088,0.076
test predictions: 6-layer: none,0.122,0.061,0.055,0.094,0.086,0.108,0.108,0.092,0.104,0.096,0.073
test predictions: 6-layer: 5NN_cosine_pca118,0.122,0.053,0.055,0.102,0.084,0.124,0.104,0.104,0.078,0.098,0.076
test predictions: 6-layer: logreg_pca118,0.124,0.045,0.051,0.092,0.092,0.129,0.104,0.102,0.082,0.098,0.082


number of the 490 test rows where two submissions differ:


,4-layer: none,4-layer: 5NN_cosine_pca118,4-layer: logreg_pca118,6-layer: none,6-layer: 5NN_cosine_pca118,6-layer: logreg_pca118
4-layer: none,0,32,33,74,72,79
4-layer: 5NN_cosine_pca118,32,0,29,80,77,88
4-layer: logreg_pca118,33,29,0,76,72,84
6-layer: none,74,80,76,0,54,63
6-layer: 5NN_cosine_pca118,72,77,72,54,0,44
6-layer: logreg_pca118,79,88,84,63,44,0
